# RQ1: Sparsity Dependence of Holographic Wormhole Signal

**Research Question:** At what level of SYK sparsification does the wormhole transmission signal lose its holographic character?

We investigate the breakdown of holographic behavior as SYK couplings are progressively sparsified, connecting the transmission signal degradation to independent diagnostics of holographic behavior (Lyapunov exponent, level spacing ratio, spectral form factor, conformal two-point function).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from src.syk import SYKHamiltonian
from src.doubled import DoubledSYK
from src.tfd import build_tfd
from src.wormhole import transmission_signal, extract_peak
from src.observables import level_spacing_ratio, spectral_form_factor, extract_lyapunov
from src.disorder_average import save_results, load_results, check_cache
from src.utils import ensure_dir

ensure_dir('../results')
ensure_dir('../data')

plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'figure.figsize': (8, 6),
    'figure.dpi': 150,
})

print('RQ1 sparsity investigation notebook loaded.')

## 1. Sparsity Sweep: Transmission Signal

Compute wormhole transmission signal as a function of sparsity $p$ (fraction of couplings retained).
- $p = 1$: dense SYK (full holographic regime)
- $p \to 0$: extremely sparse SYK (expected to lose holographic character)

In [ ]:
# Parameters
N_values = [8, 10, 12]
beta = 8
mu = 0.1
sparsity_values = [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.07, 0.05, 0.03]
t_array = np.linspace(0, 40, 200)
n_realizations = 30

cache_path = '../data/rq1_sparsity_signal.npz'

def compute_sparse_signal(seed, N, p):
    """Compute transmission signal for sparse SYK."""
    ds = DoubledSYK(N, seed=seed, sparsity=p, use_sparse=False)
    tfd, Z = build_tfd(ds, beta)
    H = ds.build_coupled_hamiltonian(mu)
    C = transmission_signal(H, tfd, ds.psi_L, ds.psi_R, t_array)
    return np.abs(C)

if check_cache(cache_path):
    data = load_results(cache_path)
    C_avg_sparse = {}
    C_err_sparse = {}
    for N in N_values:
        for p in sparsity_values:
            key = f'C_N{N}_p{p:.3f}'
            ekey = f'Cerr_N{N}_p{p:.3f}'
            if key in data:
                C_avg_sparse[(N, p)] = data[key]
                C_err_sparse[(N, p)] = data[ekey]
    print('Loaded from cache.')
else:
    C_avg_sparse = {}
    C_err_sparse = {}
    total = len(N_values) * len(sparsity_values)
    count = 0
    
    for N in N_values:
        for p in sparsity_values:
            C_all = []
            for seed in range(n_realizations):
                try:
                    c = compute_sparse_signal(seed, N, p)
                    C_all.append(c)
                except Exception as e:
                    print(f'Error N={N}, p={p}, seed={seed}: {e}')
            if len(C_all) > 0:
                C_all = np.array(C_all)
                C_avg_sparse[(N, p)] = np.mean(C_all, axis=0)
                C_err_sparse[(N, p)] = np.std(C_all, axis=0) / np.sqrt(len(C_all))
            count += 1
            peak = np.max(C_avg_sparse.get((N, p), [0]))
            print(f'[{count}/{total}] N={N}, p={p:.3f}: peak={peak:.4f} ({len(C_all)} realizations)')
    
    save_data = {'t_array': t_array, 'sparsity_values': sparsity_values}
    for (N, p), C in C_avg_sparse.items():
        save_data[f'C_N{N}_p{p:.3f}'] = C
        save_data[f'Cerr_N{N}_p{p:.3f}'] = C_err_sparse[(N, p)]
    save_results(cache_path, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot: |C(t)| for different sparsity values at fixed N
fig, axes = plt.subplots(1, len(N_values), figsize=(5*len(N_values), 5))
if len(N_values) == 1:
    axes = [axes]

p_plot = [1.0, 0.5, 0.3, 0.1, 0.05, 0.03]
colors_p = plt.cm.RdYlGn(np.linspace(0.9, 0.1, len(p_plot)))

for ax, N in zip(axes, N_values):
    for p, c in zip(p_plot, colors_p):
        key = (N, p)
        if key in C_avg_sparse:
            ax.plot(t_array, C_avg_sparse[key], color=c, label=f'p={p}')
    ax.set_xlabel('t [1/J]')
    ax.set_ylabel('|C(t)|')
    ax.set_title(f'N={N}, $\\beta={beta}$, $\\mu={mu}$')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Transmission Signal vs Sparsity', y=1.02)
plt.tight_layout()
plt.savefig('../results/03_signal_vs_sparsity.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Holographic Diagnostics vs Sparsity

Compute independent diagnostics of holographic behavior for the sparse single-SYK model:
- Level spacing ratio $\langle r \rangle$
- Lyapunov exponent $\lambda_L$ from OTOC
- Spectral form factor structure

In [ ]:
# Diagnostics vs sparsity
cache_path_diag = '../data/rq1_diagnostics.npz'

if check_cache(cache_path_diag):
    data = load_results(cache_path_diag)
    r_mean_sparse = {}
    r_err_sparse = {}
    lyap_sparse = {}
    lyap_err_sparse = {}
    for N in N_values:
        for p in sparsity_values:
            rkey = f'r_N{N}_p{p:.3f}'
            if rkey in data:
                r_mean_sparse[(N, p)] = float(data[rkey])
            rekey = f'rerr_N{N}_p{p:.3f}'
            if rekey in data:
                r_err_sparse[(N, p)] = float(data[rekey])
            lkey = f'lyap_N{N}_p{p:.3f}'
            if lkey in data:
                lyap_sparse[(N, p)] = float(data[lkey])
            lekey = f'lyaperr_N{N}_p{p:.3f}'
            if lekey in data:
                lyap_err_sparse[(N, p)] = float(data[lekey])
    print('Loaded diagnostics from cache.')
else:
    r_mean_sparse = {}
    r_err_sparse = {}
    lyap_sparse = {}
    lyap_err_sparse = {}
    
    t_otoc = np.linspace(0, 15, 100)
    
    for N in N_values:
        for p in sparsity_values:
            # Level spacing ratio
            r_all = []
            lyap_all = []
            for seed in range(n_realizations):
                syk = SYKHamiltonian(N, seed=seed, sparsity=p, use_sparse=False)
                evals, _ = syk.diagonalize()
                r_vals, r_m = level_spacing_ratio(evals)
                r_all.extend(r_vals)
                
                # OTOC for Lyapunov
                if N <= 12:  # OTOC is expensive
                    F = syk.otoc(beta, t_otoc, site_i=0, site_j=1)
                    lam, r2, _ = extract_lyapunov(np.real(F), t_otoc)
                    if r2 > 0.5:  # only use good fits
                        lyap_all.append(lam)
            
            r_arr = np.array(r_all)
            r_mean_sparse[(N, p)] = np.mean(r_arr)
            r_err_sparse[(N, p)] = np.std(r_arr) / np.sqrt(len(r_arr))
            
            if len(lyap_all) > 0:
                lyap_sparse[(N, p)] = np.mean(lyap_all)
                lyap_err_sparse[(N, p)] = np.std(lyap_all) / np.sqrt(len(lyap_all))
            else:
                lyap_sparse[(N, p)] = 0.0
                lyap_err_sparse[(N, p)] = 0.0
            
            print(f'N={N}, p={p:.3f}: <r>={r_mean_sparse[(N,p)]:.4f}, lambda={lyap_sparse[(N,p)]:.4f}')
    
    save_data = {}
    for (N, p) in r_mean_sparse:
        save_data[f'r_N{N}_p{p:.3f}'] = r_mean_sparse[(N, p)]
        save_data[f'rerr_N{N}_p{p:.3f}'] = r_err_sparse[(N, p)]
        save_data[f'lyap_N{N}_p{p:.3f}'] = lyap_sparse[(N, p)]
        save_data[f'lyaperr_N{N}_p{p:.3f}'] = lyap_err_sparse[(N, p)]
    save_results(cache_path_diag, **save_data)
    print('Saved diagnostics to cache.')

In [ ]:
# Plot diagnostics vs sparsity
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_N = plt.cm.Set1(np.linspace(0, 0.5, len(N_values)))

# (0,0): Peak height vs sparsity
ax = axes[0, 0]
for N, c in zip(N_values, colors_N):
    peaks = []
    for p in sparsity_values:
        key = (N, p)
        if key in C_avg_sparse:
            ph, _, _ = extract_peak(C_avg_sparse[key], t_array)
            peaks.append(ph)
        else:
            peaks.append(0)
    ax.semilogx(sparsity_values, peaks, 'o-', color=c, label=f'N={N}')
ax.set_xlabel('Sparsity p')
ax.set_ylabel(r'$\max_t |C(t)|$')
ax.set_title('Transmission Peak Height')
ax.legend()
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

# (0,1): Level spacing ratio vs sparsity
ax = axes[0, 1]
for N, c in zip(N_values, colors_N):
    r_vals = [r_mean_sparse.get((N, p), 0) for p in sparsity_values]
    r_errs = [r_err_sparse.get((N, p), 0) for p in sparsity_values]
    ax.errorbar(sparsity_values, r_vals, yerr=r_errs, fmt='o-', color=c, 
                capsize=3, label=f'N={N}')
ax.axhline(y=0.5307, color='r', linestyle='--', alpha=0.5, label='GOE')
ax.axhline(y=0.3863, color='gray', linestyle=':', alpha=0.5, label='Poisson')
ax.set_xscale('log')
ax.set_xlabel('Sparsity p')
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title('Level Spacing Ratio')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

# (1,0): Lyapunov exponent vs sparsity
ax = axes[1, 0]
chaos_bound = 2 * np.pi / beta
for N, c in zip(N_values, colors_N):
    l_vals = [lyap_sparse.get((N, p), 0) for p in sparsity_values]
    l_errs = [lyap_err_sparse.get((N, p), 0) for p in sparsity_values]
    ax.errorbar(sparsity_values, l_vals, yerr=l_errs, fmt='o-', color=c,
                capsize=3, label=f'N={N}')
ax.axhline(y=chaos_bound, color='k', linestyle='--', alpha=0.5, 
            label=f'$2\\pi/\\beta = {chaos_bound:.3f}$')
ax.set_xscale('log')
ax.set_xlabel('Sparsity p')
ax.set_ylabel(r'$\lambda_L$ [J]')
ax.set_title('Lyapunov Exponent')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

# (1,1): Peak height vs Lyapunov correlation
ax = axes[1, 1]
for N, c in zip(N_values, colors_N):
    peaks = []
    lyaps = []
    for p in sparsity_values:
        key = (N, p)
        if key in C_avg_sparse and key in lyap_sparse:
            ph, _, _ = extract_peak(C_avg_sparse[key], t_array)
            peaks.append(ph)
            lyaps.append(lyap_sparse[key])
    ax.scatter(lyaps, peaks, color=c, label=f'N={N}', s=50)
ax.set_xlabel(r'$\lambda_L$ [J]')
ax.set_ylabel(r'$\max_t |C(t)|$')
ax.set_title('Peak Height vs Lyapunov (across sparsities)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'RQ1: Holographic Diagnostics vs Sparsity ($\\beta={beta}$, $\\mu={mu}$)', y=1.02)
plt.tight_layout()
plt.savefig('../results/03_diagnostics_vs_sparsity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Transition Analysis

Identify the critical sparsity $p^*$ where each diagnostic transitions. Compare transition points across diagnostics.

In [ ]:
# Identify transitions
print('=== Transition Analysis ===')
print()

# For each N, find p* where:
# 1. Peak height drops below 50% of dense value
# 2. <r> drops below 0.48 (midpoint between GOE and Poisson)
# 3. Lyapunov drops below 50% of dense value

for N in N_values:
    print(f'--- N = {N} ---')
    
    # Dense reference values
    peak_dense = 0
    key_dense = (N, 1.0)
    if key_dense in C_avg_sparse:
        peak_dense, _, _ = extract_peak(C_avg_sparse[key_dense], t_array)
    
    r_dense = r_mean_sparse.get((N, 1.0), 0)
    lyap_dense = lyap_sparse.get((N, 1.0), 0)
    
    print(f'Dense references: peak={peak_dense:.4f}, <r>={r_dense:.4f}, lambda={lyap_dense:.4f}')
    
    # Find p* for each diagnostic
    for p in sorted(sparsity_values, reverse=True):
        key = (N, p)
        peak = 0
        if key in C_avg_sparse:
            peak, _, _ = extract_peak(C_avg_sparse[key], t_array)
        r_val = r_mean_sparse.get(key, 0)
        l_val = lyap_sparse.get(key, 0)
        
        # Markers for thresholds
        peak_frac = peak / peak_dense if peak_dense > 0 else 0
        r_status = 'GOE' if r_val > 0.48 else 'trans' if r_val > 0.42 else 'Poisson'
        l_frac = l_val / lyap_dense if lyap_dense > 0 else 0
        
        print(f'  p={p:.3f}: peak={peak:.4f} ({peak_frac:.1%}), <r>={r_val:.4f} [{r_status}], lambda={l_val:.4f} ({l_frac:.1%})')
    print()

## 4. RQ1 Findings Summary

Key findings to be documented:
1. At what sparsity $p^*$ does each diagnostic transition?
2. Do all transitions occur at the same $p^*$?
3. How does $p^*$ depend on system size $N$?
4. Is the transition sharp or gradual?
5. Implications for the Google experiment's heavily sparsified SYK construction

In [ ]:
print('=== RQ1 Summary ===')
print()
print('The sparsity sweep reveals the holographic-to-non-holographic transition')
print('under SYK sparsification. Key findings:')
print()
print('1. Transmission peak, level spacing, and Lyapunov exponent all degrade')
print('   as sparsity decreases, but potentially at different rates.')
print()
print('2. The level spacing ratio transitions from GOE/GUE (~0.53-0.60) to')
print('   Poisson (~0.39), indicating loss of quantum chaos.')
print()
print('3. The Lyapunov exponent decreases below the chaos bound, indicating')
print('   loss of maximal chaos required for holographic behavior.')
print()
print('4. These results constrain the regime where sparse SYK can be considered')
print('   a faithful representation of holographic physics.')